In [ ]:
# Cell 1: Environment Setup
!pip install -q datasets transformers xgboost imbalanced-learn fsspec huggingface_hub

In [ ]:
# Cell 2: Programmatic Ingestion of Financial Datasets
import pandas as pd
import numpy as np
from datasets import load_dataset

print("--- Step 1/3: Ingesting Expense Categorization Data ---")
# Streaming the transaction category dataset from Hugging Face
try:
    # Set splits configuration
    splits = {'train': 'data/train-00000-of-00001.parquet'}
    df_hf_expense = pd.read_parquet("hf://datasets/rajeshradhakrishnan/fin-transaction-category/" + splits["train"])
    print(f"Hugging Face Expense Dataset loaded. Shape: {df_hf_expense.shape}")
    print("Sample Data:")
    print(df_hf_expense.head(3))
except Exception as e:
    print(f"Error streaming from Hugging Face: {e}")

# Fetching the backup raw merchant categorizer from GitHub
github_url = "https://raw.githubusercontent.com/j-convey/BankTextCategorizer/main/data/data1.csv"
df_git_expense = pd.read_csv(github_url, encoding='latin-1')
print(f"\nGitHub Expense Dataset loaded. Shape: {df_git_expense.shape}")
print(df_git_expense.head(3))


print("\n--- Step 2/3: Ingesting Banking Chatbot Intent Data ---")
# Loading the gold-standard Banking77 corpus from Hugging Face
try:
    banking_dataset = load_dataset("mteb/banking77")
    df_banking_train = pd.DataFrame(banking_dataset['train'])
    df_banking_test = pd.DataFrame(banking_dataset['test'])
    print(f"Banking77 Train Shape: {df_banking_train.shape}, Test Shape: {df_banking_test.shape}")
    print("Sample Query:")
    print(df_banking_train.head(3))
except Exception as e:
    print(f"Error loading Banking77: {e}")


print("\n--- Step 3/3: Ingesting Tabular Threat / Fraud Modeling Data ---")
# Downloading a public GitHub mirror of the Kaggle bank transactions fraud dataset
fraud_mirror_url = "https://raw.githubusercontent.com/mehedihasanmir/Bank-Transaction-Dataset-for-Fraud-Detection/main/bank_transactions_data_2.csv"
df_fraud_raw = pd.read_csv(fraud_mirror_url)
print(f"Tabular Threat Dataset loaded. Shape: {df_fraud_raw.shape}")
print("Sample Columns:")
print(df_fraud_raw.head(3))

--- Step 1/3: Ingesting Expense Categorization Data ---
Hugging Face Expense Dataset loaded. Shape: (225, 2)
Sample Data:
             merchant  category
0  Uber 072515 SFPOOL         3
1     United Airlines         3
2        INTRST PYMNT         5

GitHub Expense Dataset loaded. Shape: (3470, 3)
   Description Category          Sub_Category
0       Arby's     Food  FastFood_Restaurants
1  Burger King     Food  FastFood_Restaurants
2   Carl's Jr.     Food  FastFood_Restaurants

--- Step 2/3: Ingesting Banking Chatbot Intent Data ---


Banking77 Train Shape: (9993, 3), Test Shape: (3076, 3)
Sample Query:
                                                text  label    label_text
0                     I am still waiting on my card?     11  card_arrival
1  What can I do if my card still hasn't arrived ...     11  card_arrival
2  I have been waiting over a week. Is the card s...     11  card_arrival

--- Step 3/3: Ingesting Tabular Threat / Fraud Modeling Data ---
Tabular Threat Dataset loaded. Shape: (2512, 16)
Sample Columns:
  TransactionID AccountID  TransactionAmount      TransactionDate  \
0      TX000001   AC00128              14.09  2023-04-11 16:29:14   
1      TX000002   AC00455             376.24  2023-06-27 16:44:19   
2      TX000003   AC00019             126.29  2023-07-10 18:16:08   

  TransactionType   Location DeviceID      IP Address MerchantID Channel  \
0           Debit  San Diego  D000380  162.198.218.92       M015     ATM   
1           Debit    Houston  D000051     13.149.61.4       M052     ATM  

In [ ]:
# Cell 3: Short-Text Preprocessing for Expense & Chatbot Data
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 1. Define distinct preprocessing functions
def clean_text_classical(text):
    """Heavy cleaning for classical statistical models (TF-IDF + SVM/NB)"""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    # Strip punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_text_transformer(text):
    """Context-preserving cleanup for Transformer-based models (DistilBERT)"""
    if not isinstance(text, str):
        return ""
    # Normalize extra whitespaces only; preserve casing and punctuation
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# 2. Process the Expense Categorization Data
print("--- Preprocessing Expense Data ---")
df_git_clean = df_git_expense.rename(
    columns={'Description': 'text', 'Category': 'label'}
).dropna()

df_git_clean['text_classical'] = df_git_clean['text'].apply(clean_text_classical)
df_git_clean['text_transformer'] = df_git_clean['text'].apply(clean_text_transformer)

# Encode text labels to integers
expense_encoder = LabelEncoder()
df_git_clean['encoded_label'] = expense_encoder.fit_transform(df_git_clean['label'])

print(f"Processed Expense Dataset: {len(df_git_clean)} rows.")
print(df_git_clean[['text', 'text_classical', 'encoded_label']].head(3))


# 3. Process the Chatbot Intent Data (Banking77)
print("\n--- Preprocessing Banking77 Chatbot Data ---")
# Banking77 is already loaded as train/test dataframes in Cell 2
df_banking_train['text_classical'] = df_banking_train['text'].apply(clean_text_classical)
df_banking_train['text_transformer'] = df_banking_train['text'].apply(clean_text_transformer)

df_banking_test['text_classical'] = df_banking_test['text'].apply(clean_text_classical)
df_banking_test['text_transformer'] = df_banking_test['text'].apply(clean_text_transformer)

print(f"Chatbot Train Set: {len(df_banking_train)} rows.")
print(df_banking_train[['text', 'text_classical', 'label']].head(3))

--- Preprocessing Expense Data ---
Processed Expense Dataset: 3470 rows.
          text text_classical  encoded_label
0       Arby's          arbys              5
1  Burger King    burger king              5
2   Carl's Jr.       carls jr              5

--- Preprocessing Banking77 Chatbot Data ---
Chatbot Train Set: 9993 rows.
                                                text  \
0                     I am still waiting on my card?   
1  What can I do if my card still hasn't arrived ...   
2  I have been waiting over a week. Is the card s...   

                                      text_classical  label  
0                      i am still waiting on my card     11  
1  what can i do if my card still hasnt arrived a...     11  
2  i have been waiting over a week is the card st...     11  


In [ ]:
# Cell 4: Tabular Preprocessing, Outlier Labeling, and SMOTE Oversampling
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

print("--- Preprocessing Tabular Threat Dataset ---")

# 1. Map the exact columns from the fraud dataset
numerical_cols = ['TransactionAmount', 'CustomerAge', 'TransactionDuration', 'LoginAttempts', 'AccountBalance']
categorical_cols = ['TransactionType', 'Channel', 'CustomerOccupation']

# Impute continuous cols with Mean
for col in numerical_cols:
    if col in df_fraud_raw.columns:
        df_fraud_raw[col] = df_fraud_raw[col].fillna(df_fraud_raw[col].mean())

# Impute categorical cols with Mode
for col in categorical_cols:
    if col in df_fraud_raw.columns:
        mode_val = df_fraud_raw[col].mode()[0]
        df_fraud_raw[col] = df_fraud_raw[col].fillna(mode_val)

# 2. Derive Target Labels dynamically using Anomaly Detection (Isolation Forest)
print("Extracting anomalies via Isolation Forest...")
iso_forest = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)

# Fit on numerical columns to identify outlier transaction footprints
features_for_anomaly = df_fraud_raw[numerical_cols]
anomaly_labels = iso_forest.fit_predict(features_for_anomaly)

# Isolation Forest outputs -1 for anomalies and 1 for normal data. Let's map it:
# 1 = Fraud (anomaly), 0 = Legitimate (normal)
df_fraud_raw['IsFraud'] = np.where(anomaly_labels == -1, 1, 0)
print(f"Class imbalance breakdown:\n{df_fraud_raw['IsFraud'].value_counts(normalize=True) * 100}")

# 3. Z-score Normalization (Scaling) and One-Hot Encoding
scaler = StandardScaler()
scaled_numerical = scaler.fit_transform(df_fraud_raw[numerical_cols])
df_numerical_scaled = pd.DataFrame(scaled_numerical, columns=numerical_cols)

# Sparse categorical encoding
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoded_categorical = encoder.fit_transform(df_fraud_raw[categorical_cols])
encoded_categorical_cols = encoder.get_feature_names_out(categorical_cols)
df_categorical_encoded = pd.DataFrame(encoded_categorical, columns=encoded_categorical_cols)

# Combine scaled and encoded datasets back together
X = pd.concat([df_numerical_scaled, df_categorical_encoded], axis=1)
y = df_fraud_raw['IsFraud']

# 4. Stratified Split (80% Train, 20% Test) to preserve class balance before SMOTE
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print(f"\nBefore SMOTE - Training Set size: {X_train.shape[0]} rows.")
print(f"Class distribution:\n{y_train.value_counts()}")

# 5. Apply SMOTE strictly to the training split to balance the classes
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"\nAfter SMOTE - Balanced Training Set size: {X_train_resampled.shape[0]} rows.")
print(f"Balanced Class distribution:\n{y_train_resampled.value_counts()}")

--- Preprocessing Tabular Threat Dataset ---
Extracting anomalies via Isolation Forest...
Class imbalance breakdown:
IsFraud
0    94.984076
1     5.015924
Name: proportion, dtype: float64

Before SMOTE - Training Set size: 2009 rows.
Class distribution:
IsFraud
0    1908
1     101
Name: count, dtype: int64

After SMOTE - Balanced Training Set size: 3816 rows.
Balanced Class distribution:
IsFraud
0    1908
1    1908
Name: count, dtype: int64


In [ ]:
# Cell 5: Tabular Fraud Detection Benchmarking
import time
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, f1_score

print("=== Track 3: Running Tabular Fraud Benchmarks ===")

# 1. Initialize classifiers
lr_model = LogisticRegression(max_iter=1000, random_state=42)
xgb_model = XGBClassifier(eval_metric='logloss', random_state=42)

# 2. Benchmark Logistic Regression
print("\nTraining Logistic Regression...")
start_time = time.time()
lr_model.fit(X_train_resampled, y_train_resampled)
lr_train_time = time.time() - start_time

# Measure inference latency line-by-line
latencies_lr = []
for i in range(len(X_test)):
    single_row = X_test.iloc[[i]]
    t0 = time.time()
    _ = lr_model.predict(single_row)
    latencies_lr.append((time.time() - t0) * 1000) # milliseconds

y_pred_lr = lr_model.predict(X_test)
lr_f1 = f1_score(y_test, y_pred_lr, average='macro')

# 3. Benchmark XGBoost
print("Training XGBoost...")
start_time = time.time()
xgb_model.fit(X_train_resampled, y_train_resampled)
xgb_train_time = time.time() - start_time

latencies_xgb = []
for i in range(len(X_test)):
    single_row = X_test.iloc[[i]]
    t0 = time.time()
    _ = xgb_model.predict(single_row)
    latencies_xgb.append((time.time() - t0) * 1000) # milliseconds

y_pred_xgb = xgb_model.predict(X_test)
xgb_f1 = f1_score(y_test, y_pred_xgb, average='macro')

# 4. Generate Operational Performance Report
print("\n================ OPERATIONAL REPORT ================")
print(f"Logistic Regression -> Train Time: {lr_train_time:.2f}s | Macro F1: {lr_f1:.4f}")
print(f"  - P50 (Median) Latency: {np.percentile(latencies_lr, 50):.2f} ms")
print(f"  - P99 (Tail) Latency:   {np.percentile(latencies_lr, 99):.2f} ms")

print(f"\nXGBoost Classifier  -> Train Time: {xgb_train_time:.2f}s | Macro F1: {xgb_f1:.4f}")
print(f"  - P50 (Median) Latency: {np.percentile(latencies_xgb, 50):.2f} ms")
print(f"  - P99 (Tail) Latency:   {np.percentile(latencies_xgb, 99):.2f} ms")

=== Track 3: Running Tabular Fraud Benchmarks ===

Training Logistic Regression...
Training XGBoost...

================ OPERATIONAL REPORT ================
Logistic Regression -> Train Time: 0.02s | Macro F1: 0.8866
  - P50 (Median) Latency: 0.99 ms
  - P99 (Tail) Latency:   2.07 ms

XGBoost Classifier  -> Train Time: 0.10s | Macro F1: 0.9392
  - P50 (Median) Latency: 1.75 ms
  - P99 (Tail) Latency:   3.80 ms


In [ ]:
# Cell 6: Expense Categorization Benchmarking (TF-IDF + Linear SVM)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

print("=== Track 1: Running Expense Categorization Benchmarks ===")

# 1. Train-test split using the heavily cleaned text from Phase 2
X_exp = df_git_clean['text_classical']
y_exp = df_git_clean['encoded_label']

X_exp_train, X_exp_test, y_exp_train, y_exp_test = train_test_split(
    X_exp, y_exp, test_size=0.20, random_state=42
)

# 2. Instantiate pipeline
expense_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1, 2))),
    ('svm', LinearSVC(random_state=42, dual=False))
])

# 3. Train model
start_time = time.time()
expense_pipeline.fit(X_exp_train, y_exp_train)
exp_train_time = time.time() - start_time

# 4. Profile individual query inference latency
latencies_exp = []
for text in X_exp_test.head(200): # Sample 200 items for faster execution profiling
    t0 = time.time()
    _ = expense_pipeline.predict([text])
    latencies_exp.append((time.time() - t0) * 1000)

y_exp_pred = expense_pipeline.predict(X_exp_test)
exp_acc = accuracy_score(y_exp_test, y_exp_pred)
exp_f1 = f1_score(y_exp_test, y_exp_pred, average='weighted')

print("\n================ OPERATIONAL REPORT ================")
print(f"TF-IDF + Linear SVM -> Train Time: {exp_train_time:.2f}s")
print(f"  - Accuracy:           {exp_acc * 100:.2f}%")
print(f"  - Weighted F1-Score:  {exp_f1:.4f}")
print(f"  - P50 Latency:        {np.percentile(latencies_exp, 50):.2f} ms")
print(f"  - P99 Latency:        {np.percentile(latencies_exp, 99):.2f} ms")

=== Track 1: Running Expense Categorization Benchmarks ===

================ OPERATIONAL REPORT ================
TF-IDF + Linear SVM -> Train Time: 0.17s
  - Accuracy:           87.75%
  - Weighted F1-Score:  0.8763
  - P50 Latency:        0.76 ms
  - P99 Latency:        1.21 ms


In [ ]:
# Cell 7: Chatbot Intent Detection Baselines
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

print("=== Track 2: Running Chatbot Intent Baselines ===")

# 1. Vectorize text features using classical cleaning
vectorizer = CountVectorizer(max_features=8000, ngram_range=(1, 2))
tfidf_transformer = TfidfTransformer()

X_train_counts = vectorizer.fit_transform(df_banking_train['text_classical'])
X_train_tfidf = tfidf_transformer.fit_transform(X_train_counts)

X_test_counts = vectorizer.transform(df_banking_test['text_classical'])
X_test_tfidf = tfidf_transformer.transform(X_test_counts)

y_bank_train = df_banking_train['label']
y_bank_test = df_banking_test['label']

# 2. Train Multinomial Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_bank_train)
y_pred_nb = nb_model.predict(X_test_tfidf)
nb_f1 = f1_score(y_bank_test, y_pred_nb, average='macro')

# 3. Train Logistic Regression
lr_bank_model = LogisticRegression(max_iter=500, random_state=42)
lr_bank_model.fit(X_train_tfidf, y_bank_train)
y_pred_bank_lr = lr_bank_model.predict(X_test_tfidf)
bank_lr_f1 = f1_score(y_bank_test, y_pred_bank_lr, average='macro')

# 4. Profile Inference Latency for Logistic Regression (Light NLP deployment candidate)
latencies_bank = []
for i in range(min(200, X_test_tfidf.shape[0])):
    single_sample = X_test_tfidf[i]
    t0 = time.time()
    _ = lr_bank_model.predict(single_sample)
    latencies_bank.append((time.time() - t0) * 1000)

print("\n================ OPERATIONAL REPORT ================")
print(f"Multinomial Naive Bayes -> Macro F1: {nb_f1:.4f}")
print(f"Logistic Regression     -> Macro F1: {bank_lr_f1:.4f}")
print(f"  - Chatbot Intent P50 Latency: {np.percentile(latencies_bank, 50):.2f} ms")
print(f"  - Chatbot Intent P99 Latency: {np.percentile(latencies_bank, 99):.2f} ms")

=== Track 2: Running Chatbot Intent Baselines ===

================ OPERATIONAL REPORT ================
Multinomial Naive Bayes -> Macro F1: 0.7843
Logistic Regression     -> Macro F1: 0.8597
  - Chatbot Intent P50 Latency: 1.07 ms
  - Chatbot Intent P99 Latency: 7.80 ms


In [ ]:
# Cell 8: Fine-Tuning DistilBERT for Banking77 Intent Detection
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from datasets import Dataset

def preprocess_function(examples):
    # Truncation and padding handled by DataCollator
    return tokenizer(examples["text"], truncation=True)

# 1. Prepare Dataset for Transformers
# Convert pandas DFs to Hugging Face Datasets
train_ds = Dataset.from_pandas(df_banking_train[['text', 'label']])
test_ds = Dataset.from_pandas(df_banking_test[['text', 'label']])

# 2. Tokenization
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=77
)

tokenized_train = train_ds.map(preprocess_function, batched=True)
tokenized_test = test_ds.map(preprocess_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    fp16=True,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
)

print("--- Starting DistilBERT Fine-Tuning ---")
trainer.train()  #🫩🫩🫩🫩

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/9993 [00:00<?, ? examples/s]

Map:   0%|          | 0/3076 [00:00<?, ? examples/s]

--- Starting DistilBERT Fine-Tuning ---


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
50,4.317844
100,4.191769
150,3.983304
200,3.791752
250,3.587121
300,3.403862
350,3.262487
400,3.154708
450,3.061363
500,2.960399


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=625, training_loss=3.4386473999023437, metrics={'train_runtime': 1931.4532, 'train_samples_per_second': 5.174, 'train_steps_per_second': 0.324, 'total_flos': 101802380633538.0, 'train_loss': 3.4386473999023437, 'epoch': 1.0})

In [ ]:
# Cell 9: Quantize the Fine-Tuned DistilBERT
import torch

# Load your trained model from the 'results' directory
model_path = "./results/checkpoint-625"
model = AutoModelForSequenceClassification.from_pretrained(model_path)

# Apply Dynamic Quantization (Converts weights from float32 to int8)
quantized_model = torch.quantization.quantize_dynamic(
    model, {torch.nn.Linear}, dtype=torch.qint8
)

# Save the quantized model
torch.save(quantized_model.state_dict(), "quantized_banking_model.pth")
print("Model quantized and saved successfully.")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

/tmp/ipykernel_56061/4059343186.py:9: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


Model quantized and saved successfully.
